# Inspect validation patches by confusion cell

For any trained run, displays N raw RGB patches per (true → predicted) cell of the confusion matrix.
Pulls predictions from `scored_candidates.parquet` (no re-inference needed) and resolves patch paths
from `patch_meta.csv`.

**Set `RUN_NAME` and `SPLIT` below.** Defaults to the 3-class run, val split.

Sort modes:
- `most_confident_wrong` (default) — surfaces the model's worst confusions, where it's confidently wrong.
- `least_confident_right` — borderline correct calls.
- `random` — uniform sample, useful for general feel of each cell.

In [ ]:
# === parameters ===
RUN_NAME = 'baseline_v2_three_class'
# 'inspected' is the cross-country held-out labeled set — already synced locally.
# To use 'val' or 'test', sync those patches first from a pod that has them:
#   rsync -avP -e 'ssh -p <PORT>' root@<HOST>:/workspace/farm-mapping/data/patches/ data/patches/
SPLIT = 'inspected'          # 'inspected' | 'val' | 'test' | 'train'
N_PER_CELL = 12
SORT_MODE = 'most_confident_wrong'   # 'most_confident_wrong' | 'least_confident_right' | 'random'
GRID_COLS = 6
CLASS_NAMES = ['NotFarm', 'Poultry', 'OtherFarm']

REPO_ROOT = '/home/filip/code/farm-mapping'
import os, sys
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# === load scored predictions + patch index ===
scored = gpd.read_parquet(f'data/output/{RUN_NAME}/scored_candidates.parquet')
meta = pd.read_csv('data/patches/patch_meta.csv')

# Latest patch per candidate (in case of duplicates)
meta = meta.drop_duplicates('candidate_id', keep='last')
df = scored.merge(meta[['candidate_id', 'patch_path']], on='candidate_id', how='left')
df['patch_exists'] = df['patch_path'].apply(
    lambda p: isinstance(p, str) and (Path('data/patches') / p).exists()
)

print(f'{RUN_NAME!r}:  scored={len(scored):,}   merged={len(df):,}   patches present locally={df.patch_exists.sum():,}')
print(f'Split sizes:')
print(df.groupby('split').size().rename('n').to_frame())

In [ ]:
# === filter to the chosen split (with patches present) ===
sub = df[(df['split'] == SPLIT) & df['patch_exists']].copy()
sub['true_label'] = sub['true_label'].astype(int)
sub['predicted_label'] = sub['predicted_label'].astype(int)
sub['correct'] = sub['true_label'] == sub['predicted_label']

if 'confidence' not in sub.columns:
    # Confidence = probability assigned to the predicted class
    prob_cols = [c for c in sub.columns if c.startswith('prob_class')]
    prob_mat = sub[prob_cols].to_numpy()
    sub['confidence'] = prob_mat[np.arange(len(sub)), sub['predicted_label'].values]

n_total = len(sub)
print(f'{SPLIT} split with patches: {n_total:,} samples')
if n_total == 0:
    print('No patches available locally for this split — rsync /workspace/farm-mapping/data/patches/ from the pod first.')

# Per-cell counts (full split, not just patches present)
split_full = df[df['split'] == SPLIT]
cm = pd.crosstab(split_full['true_label'].astype(int),
                 split_full['predicted_label'].astype(int),
                 rownames=['true'], colnames=['pred'])
cm.index = [CLASS_NAMES[i] for i in cm.index]
cm.columns = [CLASS_NAMES[i] for i in cm.columns]
print('Confusion matrix on the full split:')
cm

In [ ]:
# === patch loader + plotting helpers ===
RGB_INDICES = [2, 1, 0]          # B4, B3, B2 in our patch layout (B2,B3,B4,B8,B11,B12,...)
RGB_DIVIDE = 3000.0              # S2 SR display scaling — typical for natural color

def load_rgb(patch_path):
    arr = np.load(f'data/patches/{patch_path}').astype(np.float32)
    rgb = arr[RGB_INDICES].transpose(1, 2, 0)   # H,W,3
    rgb = np.clip(rgb / RGB_DIVIDE, 0, 1)
    return rgb

def pick(sub_cell, k, mode):
    if len(sub_cell) == 0:
        return sub_cell
    if mode == 'most_confident_wrong' and not sub_cell['correct'].iloc[0]:
        return sub_cell.nlargest(k, 'confidence')
    if mode == 'least_confident_right' and sub_cell['correct'].iloc[0]:
        return sub_cell.nsmallest(k, 'confidence')
    if mode == 'most_confident_wrong':
        # Correct cell — show most confident correct (the prototypical model wins)
        return sub_cell.nlargest(k, 'confidence')
    if mode == 'least_confident_right':
        return sub_cell.nsmallest(k, 'confidence')
    return sub_cell.sample(min(k, len(sub_cell)), random_state=42)

def show_cell(cell_df, title, color):
    n = len(cell_df)
    if n == 0:
        return
    rows = int(np.ceil(n / GRID_COLS))
    fig, axes = plt.subplots(rows, GRID_COLS, figsize=(GRID_COLS * 2.2, rows * 2.4))
    axes = np.atleast_2d(axes).flatten()
    for i, (_, r) in enumerate(cell_df.iterrows()):
        ax = axes[i]
        try:
            ax.imshow(load_rgb(r['patch_path']))
        except Exception as e:
            ax.text(0.5, 0.5, f'load err\n{e}', ha='center', va='center', fontsize=7)
        ax.set_title(
            f"{r['country']}\nconf={r['confidence']:.2f}",
            fontsize=8,
        )
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[n:]:
        ax.axis('off')
    fig.suptitle(title, fontsize=11, color=color, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# === one grid per (true, pred) cell — correct first, then confusions by size ===
K = len(CLASS_NAMES)
cells = []
for t in range(K):
    for p in range(K):
        cell_full = split_full[(split_full['true_label'].astype(int) == t) &
                                (split_full['predicted_label'].astype(int) == p)]
        cell_avail = sub[(sub['true_label'] == t) & (sub['predicted_label'] == p)]
        cells.append((t, p, len(cell_full), cell_avail))

# Order: correct first, then confusions by total count descending
cells.sort(key=lambda x: (x[0] != x[1], -x[2]))

for t, p, n_full, cell_avail in cells:
    if n_full == 0:
        continue
    sample = pick(cell_avail, N_PER_CELL, SORT_MODE)
    if t == p:
        title = f'✓  {CLASS_NAMES[t]} (correct)  —  n={n_full}, showing {len(sample)}'
        color = '#2ecc71'
    else:
        title = f'✗  true: {CLASS_NAMES[t]}  →  pred: {CLASS_NAMES[p]}  —  n={n_full}, showing {len(sample)}'
        color = '#e74c3c'
    if len(sample) == 0:
        print(f'(no patches available locally for: {title})')
        continue
    show_cell(sample, title, color)

## Drill into a single cell

Set `TRUE_CLS` and `PRED_CLS` below to inspect one cell with more samples and metadata.

In [ ]:
TRUE_CLS = 1      # Poultry
PRED_CLS = 2      # OtherFarm — the biggest leak in the 3-class run
N_DRILL = 30

drill = sub[(sub['true_label'] == TRUE_CLS) & (sub['predicted_label'] == PRED_CLS)]
drill = drill.nlargest(N_DRILL, 'confidence') if SORT_MODE == 'most_confident_wrong' \
        else drill.sample(min(N_DRILL, len(drill)), random_state=0)
print(f'{len(drill)} samples to inspect (of {((sub.true_label==TRUE_CLS)&(sub.predicted_label==PRED_CLS)).sum()} available locally)')
print('\nCountry breakdown of this cell:')
print(drill.groupby('country').size().sort_values(ascending=False).rename('n').to_frame())

show_cell(drill, f'true: {CLASS_NAMES[TRUE_CLS]}  →  pred: {CLASS_NAMES[PRED_CLS]}', '#e74c3c')